# Módulo 02 · Aula 4: Exercício: perguntas e respostas com citações

**Duração:** 45 min

## Enunciado

Você vai montar a função que responde às perguntas dos clientes da TechNova com base nos documentos, juntando as três
aulas do módulo:

```mermaid
flowchart LR
    Q([pergunta]) --> R[HybridRetriever<br/>modo à sua escolha<br/>CUSTOMERS_ONLY]
    R --> G[generate_answer<br/>RAGAnswer + citações]
    G --> A([resposta, found,<br/>fontes, uso])
```

Depois, vai rodá-la em **seis perguntas** do conjunto de avaliação (`data/eval/questions.json`) e verificar,
objetivamente, se as respostas e as fontes se comportam como um assistente honesto deve se comportar.

## Atividades

1. Criar o `retriever` com o `HybridRetriever`, escolhendo o **modo** (`"vector"`, `"bm25"` ou `"hybrid"`) e
   **justificando** a escolha com o que você mediu na aula 2.
2. Implementar `answer_question(question)`: busca + `generate_answer`, devolvendo também os trechos usados.
3. Rodar a função nas seis perguntas e montar uma tabela com resposta, fontes citadas e se a fonte esperada foi citada.
4. Rodar as verificações finais.

## Critérios de sucesso

- ✅ Toda resposta **encontrada** (`found=True`) tem pelo menos uma fonte.
- ✅ A pergunta **sem resposta** na base (`no_answer`) não inventa: `found=False` e nenhuma fonte.
- ✅ A pergunta **com dois documentos** (`multi_document`) cita pelo menos **2 fontes diferentes**.
- ✅ **Nenhuma** fonte citada é arquivada ou interna.

> Custo: cerca de 7 chamadas ao modelo de chat (uma no teste da atividade 2 e seis na atividade 3), cada uma com
> cerca de 1.500 tokens. As verificações das atividades 1 e 4 não chamam o modelo de chat.

## 0. Preparação (fornecido)

In [1]:
# Os notebooks rodam a partir da raiz do projeto: é lá que ficam o .env, a pasta data/ e o pacote src/.
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Chat e embeddings vêm do Azure OpenAI configurado no .env (veja src/config.py).
from src.config import settings

In [2]:
import pandas as pd
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever

from src.evaluation import load_questions
from src.models import add_usage
from src.rag.generation import generate_answer
from src.rag.retrieval import HybridRetriever
from src.rag.vectorstore import index_exists, load_index, stored_chunks

if not index_exists():
    from src.rag.ingestion import index_folder

    index_folder()

index = load_index()

# Fontes que NUNCA podem aparecer para um cliente: arquivadas ou de público interno (lidas dos metadados do índice).
BLOCKED_SOURCES = {
    c.metadata["source"]
    for c in stored_chunks(index).values()
    if c.metadata.get("status") != "active" or c.metadata.get("audience") != "customers"
}
print("Fontes bloqueadas:", sorted(BLOCKED_SOURCES))

# As seis perguntas do exercício: 2 fatos, 1 resposta negativa, 1 procedimento, 1 com dois documentos e 1 sem resposta.
SELECTED_IDS = ["q01", "q05", "q11", "q14", "q26", "q29"]
questions = [q for q in load_questions() if q["id"] in SELECTED_IDS]
for q in questions:
    print(f"{q['id']} [{q['type']:<14}] {q['question']}  -> esperado: {q['expected_sources'] or 'nenhuma fonte'}")

2026-09-27 21:04:36 | INFO     | src.models | Inicializando embeddings | provedor=azure | deployment=text-embedding-3-large


Fontes bloqueadas: ['internal/refund_approval_procedure.md', 'policies/shipping_2024.md']
q01 [fact          ] Qual o valor mínimo da compra pra ter frete grátis?  -> esperado: ['policies/shipping.md']
q05 [fact          ] Quanto dura a bateria do Fone Nova Buds com o cancelamento de ruído ligado?  -> esperado: ['manuals/fn-001_nova_buds.pdf']
q11 [fact          ] Vi que tem um cupom NOVA50 dando 50% de desconto em tudo. Ele funciona?  -> esperado: ['help_center/coupon_not_applied.html']
q14 [procedure     ] Quero devolver um produto pelos Correios. Como funciona o código de postagem?  -> esperado: ['help_center/return_shipping_code.html']
q26 [multi_document] É minha primeira compra. Posso usar o cupom BEMVINDO10 e ainda ganhar o desconto do Pix?  -> esperado: ['help_center/coupon_not_applied.html', 'policies/payment.md']
q29 [no_answer     ] A TechNova está contratando? Onde mando meu currículo?  -> esperado: nenhuma fonte


Repare na **q01** (frete grátis): na aula 1, sem filtro, a política de frete **arquivada** aparecia em 2º lugar para
essa pergunta. E na **q11** o cliente cita um cupom falso: a resposta certa é negativa, mas **encontrada**.

## Atividade 1: o retriever (5 min)

Crie `retriever` com `HybridRetriever(index=index, mode=...)`, mantendo o filtro padrão (`CUSTOMERS_ONLY`) e `k=4`.
Escolha o modo e escreva a justificativa em `MODE_JUSTIFICATION` (uma ou duas frases, com base nos números da aula
2: vetorial com taxa de acerto 0,962 e MRR 0,756; BM25 com 0,885 e 0,603; híbrida com 0,923 e 0,683; e o BM25
ganhando nas consultas com códigos).

<details><summary>Dica</summary>

Não existe resposta única. Pense no tipo de pergunta das seis selecionadas (linguagem natural? códigos como NOVA50 e
BEMVINDO10?) e no que os números mostraram. O filtro de metadados já vem no padrão do `HybridRetriever`: não troque
por `None`.
</details>

In [3]:
MODE = "vector"
MODE_JUSTIFICATION = (
    "Na aula 2, a vetorial teve a melhor taxa de acerto (0,962) e o melhor MRR (0,756) no conjunto de avaliação, "
    "que é escrito em linguagem natural como estas seis perguntas. Os dois cupons (NOVA50 e BEMVINDO10) aparecem em "
    "frases completas, e a vetorial também encontra o artigo de cupons nelas."
)

retriever = HybridRetriever(index=index, mode=MODE)

In [4]:
# Verificação da Atividade 1 (uma chamada de embeddings, sem LLM)
assert isinstance(retriever, BaseRetriever), "Crie o retriever com HybridRetriever(index=index, mode=...)."
assert retriever.mode in ("vector", "bm25", "hybrid"), f"Modo inválido: {retriever.mode!r}"
assert "TODO" not in MODE_JUSTIFICATION and len(MODE_JUSTIFICATION) > 30, "Escreva a justificativa da escolha."
assert retriever.metadata_filter is not None, "Mantenha o filtro de metadados (CUSTOMERS_ONLY)."
assert retriever.k == 4, "Use k=4, o padrão do projeto."

probe = retriever.invoke("Qual o valor mínimo da compra pra ter frete grátis?")
leaked = {c.metadata["source"] for c in probe} & BLOCKED_SOURCES
assert not leaked, f"O retriever devolveu fontes bloqueadas: {leaked}"
print(f"✅ Retriever no modo '{retriever.mode}', com filtro: nenhuma fonte arquivada ou interna na busca de frete.")

✅ Retriever no modo 'vector', com filtro: nenhuma fonte arquivada ou interna na busca de frete.


## Atividade 2: `answer_question` (10 min)

Implemente `answer_question(question)`:

1. busque os trechos com o `retriever`;
2. gere a resposta com `generate_answer(question, chunks)`;
3. devolva o dicionário de `generate_answer` (`answer`, `found`, `sources`, `usage`) **mais** a chave `chunks`, com os
   trechos usados (vamos precisar deles para conferir as fontes).

<details><summary>Dica</summary>

`generate_answer` já faz quase tudo: numera os trechos, chama o modelo, traduz as citações em fontes e devolve
`NO_ANSWER_MESSAGE` quando não encontra. Um `{**result, "chunks": chunks}` resolve o retorno.
</details>

In [5]:
def answer_question(question: str) -> dict:
    """Responde com base nos documentos: busca os trechos e gera a resposta com citações."""
    chunks = retriever.invoke(question)
    result = generate_answer(question, chunks)
    return {**result, "chunks": chunks}

In [6]:
# Verificação da Atividade 2 (uma chamada ao modelo)
test = answer_question("Qual o prazo para trocar um produto?")
missing = {"answer", "found", "sources", "usage", "chunks"} - set(test)
assert not missing, f"Faltam chaves no retorno: {missing}"
assert test["chunks"] and all(isinstance(c, Document) for c in test["chunks"]), "`chunks` deve ser a lista de Document usada."
assert test["found"] is True, "Essa pergunta tem resposta na base (30 dias): found deveria ser True."
assert test["sources"], "Uma resposta encontrada precisa de pelo menos uma fonte."
print(f"✅ answer_question funcionando: {test['answer'][:90]}...")
print(f"   fontes: {[s['source'] for s in test['sources']]} | uso: {test['usage']}")

2026-09-27 21:04:37 | INFO     | src.models | Inicializando LLM | provedor=azure | deployment=gpt-5.4-mini


✅ answer_question funcionando: O prazo para trocar um produto é de até 30 dias corridos após a entrega, desde que ele est...
   fontes: ['faq.csv', 'policies/exchange.md'] | uso: {'input_tokens': 983, 'output_tokens': 78, 'total_tokens': 1061, 'calls': 1}


## Atividade 3: as seis perguntas (15 min)

Rode `answer_question` em cada pergunta de `questions` e monte a lista `rows`, com um dicionário por pergunta:

| Chave | Conteúdo |
|---|---|
| `id`, `type` | da pergunta |
| `found` | do resultado |
| `cited` | lista **sem repetição** das fontes citadas (`source` de cada item de `sources`), na ordem |
| `expected` | `expected_sources` da pergunta |
| `expected_cited` | `True` se alguma fonte esperada foi citada; `None` quando a pergunta não tem fonte esperada |
| `answer` | o texto da resposta |

Guarde também os resultados completos em `results` (dicionário `id -> resultado`) e some o consumo de todas as
chamadas em `total_usage` com `add_usage`.

<details><summary>Dica</summary>

`list(dict.fromkeys(...))` remove repetidos mantendo a ordem. `add_usage(*[r["usage"] for r in results.values()])`
soma os consumos.
</details>

In [7]:
results: dict[str, dict] = {}
rows: list[dict] = []

for item in questions:
    result = answer_question(item["question"])
    results[item["id"]] = result
    cited = list(dict.fromkeys(s["source"] for s in result["sources"]))
    expected = item["expected_sources"]
    rows.append(
        {
            "id": item["id"],
            "type": item["type"],
            "found": result["found"],
            "cited": cited,
            "expected": expected,
            "expected_cited": bool(set(cited) & set(expected)) if expected else None,
            "answer": result["answer"],
        }
    )

total_usage = add_usage(*[r["usage"] for r in results.values()])

table = pd.DataFrame(rows)
table

,id,type,found,cited,expected,expected_cited,answer
0,q01,fact,True,"[faq.csv, policies/shipping.md]",[policies/shipping.md],True,"O frete é grátis para compras acima de R$ 299,..."
1,q05,fact,True,"[faq.csv, manuals/fn-001_nova_buds.pdf]",[manuals/fn-001_nova_buds.pdf],True,"Com o cancelamento de ruído ligado, a bateria ..."
2,q11,fact,True,[help_center/coupon_not_applied.html],[help_center/coupon_not_applied.html],True,Não. A TechNova informa que não existe cupom N...
3,q14,procedure,True,"[help_center/return_shipping_code.html, faq.csv]",[help_center/return_shipping_code.html],True,"Para devolver um produto pelos Correios, você ..."
4,q26,multi_document,True,"[help_center/coupon_not_applied.html, policies...","[help_center/coupon_not_applied.html, policies...",True,Sim. O cupom oficial de boas-vindas BEMVINDO10...
5,q29,no_answer,False,[],[],None,Não encontrei essa informação nos documentos d...


In [8]:
# Leitura das respostas (fornecido)
for row in rows:
    print(f"{row['id']} [{row['type']}] found={row['found']} | fonte esperada citada: {row['expected_cited']}")
    print(f"   {row['answer']}")
    print(f"   fontes: {row['cited'] or '(nenhuma)'}\n")
print("Consumo total:", total_usage)

q01 [fact] found=True | fonte esperada citada: True
   O frete é grátis para compras acima de R$ 299,00 na modalidade Normal. O valor considerado é o total dos produtos após descontos, sem contar o frete. [1][2]
   fontes: ['faq.csv', 'policies/shipping.md']

q05 [fact] found=True | fonte esperada citada: True
   Com o cancelamento de ruído ligado, a bateria do Fone Nova Buds dura até 6 horas. Com o estojo, a autonomia total chega a até 24 horas com ANC ligado. [1][2][3]
   fontes: ['faq.csv', 'manuals/fn-001_nova_buds.pdf']

q11 [fact] found=True | fonte esperada citada: True
   Não. A TechNova informa que não existe cupom NOVA50; se você viu esse código em algum lugar, ele não é da TechNova. Os cupons oficiais são divulgados apenas por e-mail cadastrado, app TechNova e site da TechNova. [2]
   fontes: ['help_center/coupon_not_applied.html']

q14 [procedure] found=True | fonte esperada citada: True
   Para devolver um produto pelos Correios, você precisa solicitar a troca ou devolução

## Atividade 4: verificações finais (5 min)

As células abaixo são fornecidas: rode e leia as mensagens. Se alguma falhar, a mensagem diz o que procurar. Os erros
mais comuns são de **busca** (o trecho certo não chegou ao modelo: experimente outro modo) ou de **tradução das
citações** (reveja a Atividade 3).

In [9]:
# Critério 1: toda resposta encontrada tem pelo menos uma fonte
assert len(rows) == 6, "Rode a Atividade 3 nas seis perguntas."
no_source = [r["id"] for r in rows if r["found"] and not r["cited"]]
assert not no_source, f"Respostas encontradas sem fonte: {no_source}"
print("✅ Toda resposta encontrada tem pelo menos uma fonte.")

✅ Toda resposta encontrada tem pelo menos uma fonte.


In [10]:
# Critério 2: a pergunta sem resposta não inventa
for r in rows:
    if r["type"] == "no_answer":
        assert r["found"] is False, f"{r['id']}: a base não responde, mas found=True. Resposta: {r['answer']}"
        assert not r["cited"], f"{r['id']}: 'não encontrei' não deve vir com fontes."
print("✅ A pergunta sem resposta foi admitida, sem fontes.")

✅ A pergunta sem resposta foi admitida, sem fontes.


In [11]:
# Critério 3: a pergunta com dois documentos cita pelo menos 2 fontes diferentes
for r in rows:
    if r["type"] == "multi_document":
        assert len(r["cited"]) >= 2, (
            f"{r['id']} citou {r['cited']}. Ela precisa de {r['expected']}: confira se os dois documentos "
            "chegaram nos trechos (results[id]['chunks']) antes de culpar o modelo."
        )
print("✅ A pergunta com dois documentos cita fontes diferentes.")

✅ A pergunta com dois documentos cita fontes diferentes.


In [12]:
# Critério 4: nenhuma fonte citada é arquivada ou interna (e nenhum trecho usado também)
cited_blocked = {s for r in rows for s in r["cited"]} & BLOCKED_SOURCES
used_blocked = {c.metadata["source"] for res in results.values() for c in res["chunks"]} & BLOCKED_SOURCES
assert not cited_blocked, f"Fontes bloqueadas citadas: {cited_blocked}"
assert not used_blocked, f"Trechos bloqueados chegaram ao modelo: {used_blocked}"
print("✅ Nenhuma fonte arquivada ou interna, nem nas citações nem no contexto.")

✅ Nenhuma fonte arquivada ou interna, nem nas citações nem no contexto.


In [13]:
# Informativo (não é critério): a fonte esperada foi citada?
with_expected = [r for r in rows if r["expected_cited"] is not None]
hits = sum(r["expected_cited"] for r in with_expected)
print(f"Fonte esperada citada em {hits} de {len(with_expected)} perguntas com fonte esperada.")
for r in with_expected:
    if not r["expected_cited"]:
        print(f"  {r['id']}: citou {r['cited']}, esperado {r['expected']}. A resposta está errada ou só veio de outra fonte?")

Fonte esperada citada em 5 de 5 perguntas com fonte esperada.


> 💡 Quando a fonte esperada não é citada, leia a resposta antes de concluir que ela está errada: muitas vezes a
> pergunta frequente (`faq.csv`) responde a mesma coisa que a política. Essa diferença entre "a fonte do gabarito" e "uma
> fonte correta" é um dos temas da avaliação no módulo 03.

## Desafio opcional

O front-end mostra as fontes embaixo da resposta. Implemente `format_for_customer(result)`, que devolve um texto com a
resposta e, se houver fontes, uma seção "Fontes:" com uma linha por fonte no formato
`[n] Título (arquivo, página p)`, omitindo a página quando ela não existe. Sem fontes (resposta não encontrada), devolva
só a resposta. Sem chamadas ao modelo: use os `results` da Atividade 3.

Para ir além: troque o modo do retriever, rode as seis perguntas de novo e compare quantas vezes a fonte esperada foi
citada e o consumo total. A diferença justifica a sua escolha da Atividade 1?

In [14]:
def format_for_customer(result: dict) -> str:
    """Resposta pronta para exibir: o texto e, se houver, as fontes numeradas com título, arquivo e página."""
    if not result["sources"]:
        return result["answer"]
    lines = [result["answer"], "", "Fontes:"]
    for s in result["sources"]:
        page = f", página {s['page']}" if s.get("page") else ""
        lines.append(f"[{s['number']}] {s['title']} ({s['source']}{page})")
    return "\n".join(lines)


for qid in ["q05", "q29"]:
    print(format_for_customer(results[qid]))
    print("-" * 60)

assert format_for_customer(results["q29"]) == results["q29"]["answer"], "Sem fontes, devolva só a resposta."
assert "Fontes:" in format_for_customer(results["q05"])
print("✅ Formatação para o cliente.")

Com o cancelamento de ruído ligado, a bateria do Fone Nova Buds dura até 6 horas. Com o estojo, a autonomia total chega a até 24 horas com ANC ligado. [1][2][3]

Fontes:
[1] Qual a autonomia da bateria do Fone Nova Buds? (faq.csv)
[2] Manual do usuário: Fone Nova Buds (manuals/fn-001_nova_buds.pdf, página 3)
[3] Manual do usuário: Fone Nova Buds (manuals/fn-001_nova_buds.pdf, página 2)
------------------------------------------------------------
Não encontrei essa informação nos documentos da TechNova. Se quiser, posso encaminhar sua dúvida para um atendente.
------------------------------------------------------------
✅ Formatação para o cliente.


## Resumo

- `answer_question` é a menor versão útil do assistente: **busca filtrada** (`HybridRetriever` com `CUSTOMERS_ONLY`) +
  **geração com citações** (`generate_answer`).
- A escolha do modo de busca é uma decisão de engenharia apoiada em **medição**, não em preferência.
- Um assistente honesto se verifica com critérios objetivos: resposta encontrada tem fonte; sem resposta, admite e não
  mostra fontes; perguntas compostas citam os documentos necessários; nada arquivado ou interno aparece.
- Quando algo falha, olhe primeiro os **trechos** (`chunks`): a maioria dos erros de RAG é de busca, não de geração.

## Próxima aula

No módulo 03 este pipeline vira um **grafo LangGraph**. Na aula `03_01_rag_as_a_graph` vamos definir o estado do RAG e
transformar busca e geração em nós; em seguida, o RAG corretivo avalia os trechos, reescreve a consulta quando a busca
falha e desiste com segurança.